In [1]:
"""Build architecture-aware stay-away step-level error-analysis workbooks.

This script reads the cached model results for the balanced transformer run and
writes false-positive, false-negative, and combined qualitative Excel workbooks
for DOMESTIC_STAY_AWAY_ORDER_IMPOSED.  The output is designed for qualitative
review: each row includes the text the model had available, the target
proceeding text, defendant/case metadata, step-level true/predicted labels, and
current proceeding charges.
"""

from __future__ import annotations

from collections import Counter
import pickle
import sys
from pathlib import Path
from typing import Any

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

project_root = "/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

BASE_CACHE = Path("/data/krosen5/cache/transformers/balanced_8")
MODELS_DIR = BASE_CACHE / "saved_models"
OUTPUT_DIR = BASE_CACHE / "error_analysis_v2"
TARGET = "DOMESTIC_STAY_AWAY_ORDER_IMPOSED"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _load_global_charge_names() -> dict[str, str]:
    candidates = [
        BASE_CACHE / "exported_models" / "charge_vocab.pkl",  # export_model.py
        BASE_CACHE / "temp_vocabs.pkl",  # contains (charge, mag) tuple
    ]
    for path in candidates:
        if not path.exists():
            continue
        try:
            with open(path, "rb") as f:
                obj = pickle.load(f)
            charge_vocab = obj[0] if isinstance(obj, tuple) else obj
            orig2idx = getattr(charge_vocab, "orig2idx", {}) or {}
            idx2name = getattr(charge_vocab, "idx2name", {}) or {}
            lookup = {}
            for orig_id, idx in orig2idx.items():
                name = idx2name.get(idx)
                if name:
                    lookup[str(orig_id)] = str(name)
            if lookup:
                print(f"✅ Loaded charge name lookup ({len(lookup)} charges) from {path}")
                return lookup
        except Exception as e:  # pragma: no cover - diagnostic-only path
            print(f"(warn) Could not load charge vocab from {path}: {e}")
    print("⚠️  No charge vocab found — charges will show as 'Charge {id}'")
    return {}


GLOBAL_CHARGE_NAMES = _load_global_charge_names()

# Matches RACE_MAP / GENDER_MAP in data/sequence_builder.py.
RACE_ID_TO_LABEL = {
    0: "Black",
    1: "White",
    2: "Hispanic",
    3: "Asian",
    4: "Indigenous",
    5: "Unknown",
}
GENDER_ID_TO_LABEL = {0: "Male", 1: "Female", 2: "Unknown"}


def load_pickle(path: Path) -> Any:
    with open(path, "rb") as f:
        return pickle.load(f)


def get_proc_text(proc: dict[str, Any]) -> str:
    for key in ("text", "proceeding_text", "mag_text", "cdc_text", "notes"):
        value = proc.get(key, "")
        if value and str(value).strip():
            return str(value).strip()
    return "[no text]"


def get_proc_meta(proc: dict[str, Any]) -> dict[str, Any]:
    return {
        "date": str(proc.get("date", "")),
        "proc_type": proc.get("type", ""),
        "case_id": proc.get("case_id", "") or proc.get("docket_id", ""),
        "def_id": proc.get("defendant_id", ""),
    }


def format_step(i: int, proc: dict[str, Any]) -> str:
    meta = get_proc_meta(proc)
    text = get_proc_text(proc)
    return f"[Step {i} | {meta['date']} | {meta['proc_type']}]\n{text}"


def build_context_text(sequence: list[dict[str, Any]], up_to_exclusive: int) -> str:
    """Text the model saw before predicting step ``up_to_exclusive``."""
    if up_to_exclusive == 0:
        return (
            "[No prior proceedings — model predicted Step 0 from initial state only "
            "(case metadata, defendant demographics, charge type)]"
        )
    parts = [format_step(i, sequence[i]) for i in range(up_to_exclusive)]
    return "\n\n---\n\n".join(parts)


def build_full_text(sequence: list[dict[str, Any]]) -> str:
    return "\n\n---\n\n".join(format_step(i, p) for i, p in enumerate(sequence))


def _build_charge_name_lookup(sequence: list[dict[str, Any]]) -> dict[str, str]:
    """Return charge_id -> name, preferring per-case names over global names."""
    name_lookup = dict(GLOBAL_CHARGE_NAMES)
    for p in sequence or []:
        for cid, info in (p.get("charges", {}) or {}).items():
            if isinstance(info, dict) and info.get("name"):
                name_lookup[str(cid)] = str(info["name"])
    return name_lookup


def format_charges(charges_dict: dict[Any, Any] | None, name_lookup: dict[str, str] | None = None) -> str:
    """Render a charges dict as readable charge names/counts/bonds."""
    if not charges_dict:
        return "None"

    name_lookup = name_lookup or {}
    parts = []
    for cid, info in charges_dict.items():
        count = 1
        bond = None
        name = name_lookup.get(str(cid), f"Charge {cid}")

        if isinstance(info, dict):
            count = info.get("count", 1)
            bond = info.get("bond")
            name = str(info.get("name") or name)
        else:
            try:
                count = int(info)
            except (TypeError, ValueError):
                count = 1

        piece = f"{name} (x{count})"
        if bond not in (None, "", "nan"):
            piece += f" [bond={bond}]"
        parts.append(piece)
    return " | ".join(parts)


def format_prior_charges(
    prior_charges_dict: dict[Any, Any] | None,
    name_lookup: dict[str, str] | None = None,
) -> str:
    """Render the ``all_prior_charges`` dict as a readable string."""
    return format_charges(prior_charges_dict, name_lookup)


def get_defendant_info(sequence: list[dict[str, Any]]) -> dict[str, Any]:
    """Pull case-level defendant info from sequence[0]."""
    blank = {
        "defendant_id": "",
        "defendant_race": "Unknown",
        "defendant_gender": "Unknown",
        "defendant_age": "",
        "prior_case_count": "",
        "prior_charges": "",
    }
    if not sequence:
        return blank

    proc0 = sequence[0]
    race_id = proc0.get("race_id", 5)
    gender_id = proc0.get("gender_id", 2)
    age = proc0.get("age", "")
    if age is None:
        age = ""

    name_lookup = _build_charge_name_lookup(sequence)
    return {
        "defendant_id": proc0.get("defendant_id", "") or "",
        "defendant_race": RACE_ID_TO_LABEL.get(race_id, "Unknown"),
        "defendant_gender": GENDER_ID_TO_LABEL.get(gender_id, "Unknown"),
        "defendant_age": age,
        "prior_case_count": proc0.get("prior_case_count", 0),
        "prior_charges": format_prior_charges(proc0.get("all_prior_charges", {}), name_lookup),
    }


def normalize_label_list(value: Any) -> list[str]:
    """Convert list-like/string label values from reports into a label list."""
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass
    if isinstance(value, (list, tuple, set)):
        return [str(v).strip() for v in value if str(v).strip()]
    text = str(value).strip()
    if not text:
        return []
    if " | " in text:
        return [part.strip() for part in text.split("|") if part.strip()]
    return [text]


def format_label_value(value: Any) -> str:
    return " | ".join(normalize_label_list(value))


PALETTE = {
    "FP": ("8B0000", "FDECEA"),
    "FN": ("7B4F00", "FFF8E1"),
    "Qualitative": ("4A235A", "F5EEF8"),
}

WIDE = {
    "input_context_text": 85,
    "target_step_text": 80,
    "all_proceedings_text": 90,
    "architecture_note": 70,
    "proceeding_true_labels": 55,
    "proceeding_predicted_labels": 55,
    "proceeding_predicted_confidences": 35,
    "current_charges": 70,
    "case_true_labels": 50,
    "case_pred_labels": 50,
    "case_fp_labels": 50,
    "case_fn_labels": 50,
    "error_type": 55,
    "prior_charges": 60,
    "defendant_race": 18,
    "defendant_gender": 18,
    "defendant_age": 14,
    "prior_case_count": 18,
    "defendant_id": 22,
}


def write_wb(rows: list[dict[str, Any]], path: Path, key: str, title: str) -> None:
    if not rows:
        print(f"  ⚠️  {key}: no rows — skipping")
        return
    hdr, alt = PALETTE[key]
    wb = Workbook()
    ws = wb.active
    ws.title = key
    cols = list(rows[0].keys())

    ws.merge_cells(f"A1:{get_column_letter(len(cols))}1")
    ws["A1"] = title
    ws["A1"].font = Font(name="Arial", bold=True, size=13, color=hdr)
    ws["A1"].alignment = Alignment(horizontal="center")
    ws.row_dimensions[1].height = 24
    ws.append([])

    ws.append(cols)
    header_row = ws.max_row
    for c, col in enumerate(cols, 1):
        cell = ws.cell(row=header_row, column=c)
        cell.font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
        cell.fill = PatternFill("solid", start_color=hdr)
        cell.alignment = Alignment(horizontal="center")
    ws.freeze_panes = ws.cell(row=header_row + 1, column=1)

    for i, row in enumerate(rows):
        ws.append([row.get(c, "") for c in cols])
        data_row = ws.max_row
        fill = PatternFill("solid", start_color=alt) if i % 2 == 1 else None
        for c, col in enumerate(cols, 1):
            cell = ws.cell(row=data_row, column=c)
            cell.font = Font(name="Arial", size=9)
            if fill:
                cell.fill = fill
            if col in WIDE:
                cell.alignment = Alignment(wrap_text=True, vertical="top")
        ws.row_dimensions[data_row].height = 100

    for c, col in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(c)].width = WIDE.get(col, 16)

    wb.save(path)
    print(f"  ✅ {key:14s} {len(rows):4d} rows  →  {path.name}")


def main() -> None:
    print("=" * 70)
    print(f"  Step-Level Error Analysis (architecture-aware): {TARGET}")
    print(__doc__)
    print("=" * 70)

    mr = load_pickle(MODELS_DIR / "model_results.pkl")
    fold_results = mr["cv_training_results"]["fold_results"]
    case_lvl_dict = mr["case_level_dict"]
    misclass_dict = mr["confident_misclass_results_per_fold"]

    fp_rows: list[dict[str, Any]] = []
    fn_rows: list[dict[str, Any]] = []

    for fold_idx, fold_data in enumerate(fold_results):
        val_data = fold_data["val_data"]
        case_dicts = case_lvl_dict[fold_idx]

        mc_data = misclass_dict[fold_idx]
        mc_df = None
        used_key = None
        for df_key in ("interpreted_mistakes", "mistakes_report"):
            df = mc_data.get(df_key)
            if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
                mc_df = df.copy()
                used_key = df_key
                break

        if mc_df is None or mc_df.empty:
            print(f"  Fold {fold_idx + 1}: no misclass data — skipping")
            continue

        print(f"\n── Fold {fold_idx + 1} [{used_key}] | val={len(val_data)} | mistakes={len(mc_df)} ──")
        print(f"   Columns: {list(mc_df.columns)}")

        text_col = next((c for c in ("proceeding_text", "text", "mag_text", "cdc_text") if c in mc_df.columns), None)
        case_idx_col = next((c for c in ("case_idx", "val_idx", "sequence_idx") if c in mc_df.columns), None)
        step_idx_col = next((c for c in ("step_idx", "proceeding_idx", "target_proceeding_idx", "proc_idx") if c in mc_df.columns), None)
        conf_col = next((c for c in ("wrong_label_confidence", "confidence", "prob", "prediction_prob") if c in mc_df.columns), None)
        wrong_col = next((c for c in ("wrong_label", "predicted_label", "label") if c in mc_df.columns), None)
        true_col = next((c for c in ("true_labels", "true_label", "true") if c in mc_df.columns), None)
        pred_col = next((c for c in ("predicted_labels", "pred_labels", "predicted") if c in mc_df.columns), None)
        pred_conf_col = next((c for c in ("confidences", "predicted_confidences") if c in mc_df.columns), None)

        fp_mask = pd.Series(False, index=mc_df.index)
        if wrong_col:
            fp_mask = mc_df[wrong_col].astype(str) == TARGET

        # NOTE: This is the legacy confident-mistake FN heuristic. These rows mean
        # TARGET was true while another label was confidently wrong; use the new
        # proceeding_* columns to verify whether TARGET was actually absent from
        # the top-k proceeding-level predictions.
        fn_mask = pd.Series(False, index=mc_df.index)
        if true_col:
            fn_mask = mc_df[true_col].astype(str).str.contains(TARGET, na=False)
            if wrong_col:
                fn_mask = fn_mask & (mc_df[wrong_col].astype(str) != TARGET)

        fp_steps = mc_df[fp_mask].copy()
        fn_steps = mc_df[fn_mask].copy()
        print(f"   FPs for {TARGET}: {len(fp_steps)}")
        print(f"   FNs for {TARGET}: {len(fn_steps)}")

        def enrich(step_df: pd.DataFrame, is_fp: bool) -> list[dict[str, Any]]:
            out = []
            for _, srow in step_df.iterrows():
                r = dict(srow)

                t = r.get(step_idx_col, None) if step_idx_col else None
                if t is not None:
                    try:
                        t = int(t)
                    except (ValueError, TypeError):
                        t = None

                confidence = r.get(conf_col, float("nan")) if conf_col else float("nan")
                wrong_pred = r.get(wrong_col, "") if wrong_col else ""
                true_lbl = r.get(true_col, "") if true_col else ""
                pred_lbl = r.get(pred_col, "") if pred_col else ""
                pred_conf = r.get(pred_conf_col, "") if pred_conf_col else ""
                proceeding_true_labels = format_label_value(true_lbl)
                proceeding_predicted_labels = format_label_value(pred_lbl)
                proceeding_predicted_confidences = (
                    " | ".join(str(v) for v in pred_conf)
                    if isinstance(pred_conf, (list, tuple))
                    else str(pred_conf or "")
                )
                target_true_at_proceeding = TARGET in normalize_label_list(true_lbl)
                target_predicted_at_proceeding = (
                    TARGET in normalize_label_list(pred_lbl) or str(wrong_pred) == TARGET
                )

                c_idx = r.get(case_idx_col, None) if case_idx_col else None
                if c_idx is not None:
                    try:
                        c_idx = int(c_idx)
                    except (ValueError, TypeError):
                        c_idx = None

                sequence: list[dict[str, Any]] = []
                full_text = "[case not linked]"
                case_f1 = float("nan")
                case_true = str(true_lbl)
                case_pred = ""
                case_fps = ""
                case_fns = ""
                case_id = ""
                num_steps: int | str = "?"
                current_charges = "[case not linked]"

                if c_idx is not None and c_idx < len(val_data):
                    seq_item = val_data[c_idx]
                    sequence = seq_item[0] if isinstance(seq_item, (tuple, list)) else seq_item
                    full_text = build_full_text(sequence)
                    num_steps = len(sequence)
                    if sequence:
                        case_id = get_proc_meta(sequence[0]).get("case_id", "")

                    if t is not None and sequence and t < len(sequence):
                        name_lookup = _build_charge_name_lookup(sequence)
                        current_charges = format_charges(sequence[t].get("charges", {}), name_lookup)
                    elif sequence:
                        current_charges = "[step index unavailable or out of range]"

                    matching = [cd for cd in case_dicts if cd.get("case_idx") == c_idx]
                    if matching:
                        cd = matching[0]
                        case_f1 = round(float(cd.get("f1", float("nan"))), 4)
                        case_true = " | ".join(cd.get("true_labels", []))
                        case_pred = " | ".join(cd.get("pred_labels", []))
                        case_fps = " | ".join(cd.get("false_positives", []))
                        case_fns = " | ".join(cd.get("false_negatives", []))

                def_info = get_defendant_info(sequence)

                if t is not None and sequence:
                    input_context_text = build_context_text(sequence, up_to_exclusive=t)
                    target_step_text = format_step(t, sequence[t]) if t < len(sequence) else "[step index out of range]"
                elif text_col:
                    input_context_text = "[step index unavailable — see target_step_text]"
                    target_step_text = str(r.get(text_col, "[no text]"))
                else:
                    input_context_text = "[unavailable]"
                    target_step_text = "[unavailable]"

                if t is not None:
                    step_desc = "Initial → Step 0" if t == 0 else f"Step {t - 1} → Step {t}"
                else:
                    step_desc = "?"

                if is_fp:
                    seen_range = "no prior proceeding text" if t == 0 else f"proceedings 0..{t - 1}"
                    arch_note = (
                        f"FP at {step_desc}: model saw {seen_range} and WRONGLY predicted "
                        f"{TARGET} at step {t}. input_context_text shows WHAT THE MODEL SAW "
                        f"that triggered the false prediction. target_step_text confirms no stay-away at step {t}."
                    )
                elif t == 0:
                    arch_note = (
                        f"FN at {step_desc}: {TARGET} was at Step 0 (first appearance) but model had "
                        "NO prior proceedings to predict from — only initial case metadata "
                        "(charge type, defendant demographics). input_context_text is EMPTY. "
                        "target_step_text has the stay-away language."
                    )
                else:
                    arch_note = (
                        f"FN at {step_desc}: {TARGET} was at step {t} but model, seeing only "
                        f"proceedings 0..{t - 1}, missed it. input_context_text shows what the "
                        "model saw. target_step_text has the stay-away language the model could not yet see."
                    )

                out.append(
                    {
                        "error_type": (
                            "FALSE POSITIVE — model predicted stay-away; none imposed"
                            if is_fp
                            else "FALSE NEGATIVE — stay-away imposed; model missed it"
                        ),
                        "fold": fold_idx + 1,
                        "case_idx": c_idx,
                        "case_id": case_id,
                        "num_proceedings": num_steps,
                        "defendant_id": def_info["defendant_id"],
                        "defendant_race": def_info["defendant_race"],
                        "defendant_gender": def_info["defendant_gender"],
                        "defendant_age": def_info["defendant_age"],
                        "prior_case_count": def_info["prior_case_count"],
                        "prior_charges": def_info["prior_charges"],
                        "current_charges": current_charges,
                        "prediction_target": step_desc,
                        "model_confidence": round(float(confidence), 4) if not pd.isna(confidence) else "",
                        "wrong_prediction": wrong_pred,
                        "proceeding_true_labels": proceeding_true_labels,
                        "proceeding_predicted_labels": proceeding_predicted_labels,
                        "proceeding_predicted_confidences": proceeding_predicted_confidences,
                        "target_true_at_proceeding": target_true_at_proceeding,
                        "target_predicted_at_proceeding": target_predicted_at_proceeding,
                        "architecture_note": arch_note,
                        "input_context_text": input_context_text,
                        "target_step_text": target_step_text,
                        "case_f1": case_f1,
                        "case_true_labels": case_true,
                        "case_pred_labels": case_pred,
                        "case_fp_labels": case_fps,
                        "case_fn_labels": case_fns,
                        "all_proceedings_text": full_text,
                    }
                )
            return out

        fp_rows.extend(enrich(fp_steps, is_fp=True))
        fn_rows.extend(enrich(fn_steps, is_fp=False))

    if not fp_rows and not fn_rows:
        print("\n⚠️  Both lists empty — column diagnostic:")
        for fi in range(5):
            mc_data = misclass_dict[fi]
            for key in ("interpreted_mistakes", "mistakes_report"):
                df = mc_data.get(key)
                if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
                    print(f"\n  Fold {fi + 1} [{key}] columns: {list(df.columns)}")
                    print(f"  Row 0: {df.iloc[0].to_dict()}")
                    break
        sys.exit(0)

    def _conf_sort_key(r: dict[str, Any]) -> float:
        c = r.get("model_confidence", "")
        try:
            return -float(c)
        except (TypeError, ValueError):
            return 0.0

    fp_rows.sort(key=_conf_sort_key)
    fn_rows.sort(key=_conf_sort_key)
    qual_rows = sorted(fp_rows + fn_rows, key=_conf_sort_key)

    print(f"\n{'─' * 70}")
    write_wb(
        fp_rows,
        OUTPUT_DIR / "stay_away_FP_step_level.xlsx",
        "FP",
        f"{TARGET} — False Positives | input_context_text = what model SAW (triggered FP)",
    )
    write_wb(
        fn_rows,
        OUTPUT_DIR / "stay_away_FN_step_level.xlsx",
        "FN",
        f"{TARGET} — False Negatives | target_step_text = where stay-away is; input often empty (Step 0)",
    )
    write_wb(
        qual_rows,
        OUTPUT_DIR / "stay_away_qualitative_step_level.xlsx",
        "Qualitative",
        f"All Errors — {TARGET} | Sorted by model confidence | See architecture_note column",
    )

    print(f"\n{'=' * 70}")
    print(f"  Output : {OUTPUT_DIR}")
    print(f"  FP rows: {len(fp_rows)}  |  FN rows: {len(fn_rows)}")
    print(
        """
  KEY COLUMNS:
    current_charges                  — charges attached to the target proceeding t
    proceeding_true_labels           — true regex labels at the target proceeding t
    proceeding_predicted_labels      — top-k model labels at the target proceeding t
    proceeding_predicted_confidences — confidences aligned with proceeding_predicted_labels
    target_true_at_proceeding        — whether TARGET is in proceeding_true_labels
    target_predicted_at_proceeding   — whether TARGET is in proceeding_predicted_labels or wrong_prediction
    input_context_text               — proceedings 0..t-1 the model SAW before predicting step t
    target_step_text                 — text AT step t (the prediction target)
"""
    )
    print("=" * 70)

    for label, rows in [("FP", fp_rows), ("FN", fn_rows)]:
        if not rows:
            print(f"  {label}: no rows")
            continue
        step_counts = Counter(r.get("prediction_target", "?") for r in rows)
        print(f"\n  {label} ({len(rows)} total):")
        for step, count in sorted(step_counts.items(), key=lambda x: x[1], reverse=True):
            pct = 100 * count / len(rows)
            flag = "  <- cold start (no prior text)" if "Initial" in step else ""
            print(f"    {step}: {count} ({pct:.0f}%){flag}")


if __name__ == "__main__":
    main()


✅ Loaded charge name lookup (2410 charges) from /data/krosen5/cache/transformers/balanced_8/exported_models/charge_vocab.pkl
  Step-Level Error Analysis (architecture-aware): DOMESTIC_STAY_AWAY_ORDER_IMPOSED
Build architecture-aware stay-away step-level error-analysis workbooks.

This script reads the cached model results for the balanced transformer run and
writes false-positive, false-negative, and combined qualitative Excel workbooks
for DOMESTIC_STAY_AWAY_ORDER_IMPOSED.  The output is designed for qualitative
review: each row includes the text the model had available, the target
proceeding text, defendant/case metadata, step-level true/predicted labels, and
current proceeding charges.


── Fold 1 [interpreted_mistakes] | val=507 | mistakes=679 ──
   Columns: ['case_id', 'case_idx', 'proceeding_id', 'step_idx', 'true_labels', 'predicted_labels', 'confidences', 'wrong_label', 'wrong_label_confidence', 'sequence_length', 'num_true_labels', '_sequence', '_teacher_forced_sequence', '_tr